# HW14:
# эмбеддинги, FAISS, оценка retrieval и mini-RAG по базе знаний

# Подключение библиотек и настройка среды

In [25]:
import os
import random
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
from typing import List, Dict
import warnings
warnings.filterwarnings('ignore')

# Фиксация воспроизводимости
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ARTIFACTS_DIR = "artifacts"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

# Загрузка учебной базы знаний

In [26]:
raw_documents = [
    {"title": "Сброс пароля", "content": "Для сброса пароля от корпоративной почты перейдите на portal.company.com/reset. Введите логин, получите письмо на резервную почту и следуйте инструкциям. Сброс доступен 24/7."},
    {"title": "Настройка VPN", "content": "Установите клиент OpenVPN. Скачайте конфигурационный файл с интранета. При подключении используйте логин/пароль от AD. Для работы из-за рубежа требуется согласование с ИБ."},
    {"title": "Политика удаленной работы", "content": "Удаленная работа возможна до 2 дней в неделю по согласованию с руководителем. Необходимо использовать корпоративный ноутбук и подключаться через VPN. Личные устройства запрещены."},
    {"title": "Заявка на оборудование", "content": "Новое оборудование заказывается через Jira-проект IT-PROCUREMENT. Укажите модель, обоснование и срок. Среднее время доставки — 5 рабочих дней. Для срочных случаев используйте канал urgent-hardware."},
    {"title": "Резервное копирование", "content": "Бэкапы рабочих станций выполняются ежедневно в 03:00 по расписанию Veeam. Пользователи не должны отключать машины от сети ночью. Восстановление файлов доступно через Self-Service Portal."},
    {"title": "Правила информационной безопасности", "content": "Запрещено передавать учетные данные третьим лицам. Используйте менеджер паролей. При утере устройства немедленно сообщите в Security Ops. Регулярные тренинги по фишингу обязательны раз в квартал."},
    {"title": "Техподдержка 1-й линии", "content": "Для обращения в техподдержку создайте тикет в Service Desk или позвоните по внутреннему номеру 4400. Режим работы: Пн-Пт 09:00-18:00. Критические инциденты обрабатываются вне очереди."},
    {"title": "Настройка принтеров", "content": "Сетевые принтеры добавляются автоматически через групповые политики. Если принтер не появился, запустите скрипт sync_printers.bat от имени администратора. Расходные материалы заказывает офис-менеджер."},
    {"title": "Доступ к базам данных", "content": "Доступ к prod-базам строго ограничен. Для dev/staging сред запросите доступ через Jira IT-DBA. Укажите имя базы, тип доступа и обоснование. Ключи доступа истекают каждые 90 дней."},
    {"title": "Обновление ПО", "content": "Обновления ОС и софта пушатся через WSUS и Intune. Перезагрузка может быть запланирована на выходные. Откладывать обновления можно не более 14 дней. Критические патчи применяются принудительно."}
]

print(f"Загружено документов: {len(raw_documents)}")
for i, doc in enumerate(raw_documents[:3]):
    print(f"\nПример {i+1}: '{doc['title']}'")
    print(f"   Длина: {len(doc['content'])} символов")
    print(f"   Начало: {doc['content'][:60]}...")

Загружено документов: 10

Пример 1: 'Сброс пароля'
   Длина: 174 символов
   Начало: Для сброса пароля от корпоративной почты перейдите на portal...

Пример 2: 'Настройка VPN'
   Длина: 171 символов
   Начало: Установите клиент OpenVPN. Скачайте конфигурационный файл с ...

Пример 3: 'Политика удаленной работы'
   Длина: 178 символов
   Начало: Удаленная работа возможна до 2 дней в неделю по согласованию...


# Реализация чанкинга и пример разбиения

In [27]:
def chunk_text(text: str, chunk_size: int = 150, overlap: int = 30) -> List[str]:
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk_words = words[start:end]
        if not chunk_words: break
        chunks.append(" ".join(chunk_words))
        start = end - overlap
    return chunks

CHUNK_SIZE = 150
CHUNK_OVERLAP = 30

all_chunks = []
for doc in raw_documents:
    doc_chunks = chunk_text(doc["content"], CHUNK_SIZE, CHUNK_OVERLAP)
    for chunk in doc_chunks:
        all_chunks.append({
            "source_title": doc["title"],
            "text": chunk
        })

print(f"Всего чанков после разбиения: {len(all_chunks)}")
print(f"\nПример '{raw_documents[0]['title']}':")
sample_chunks = [c for c in all_chunks if c["source_title"] == raw_documents[0]["title"]]
for i, c in enumerate(sample_chunks):
    print(f"  Чанк {i+1}: {c['text'][:60]}... ({len(c['text'].split())} слов)")

Всего чанков после разбиения: 10

Пример 'Сброс пароля':
  Чанк 1: Для сброса пароля от корпоративной почты перейдите на portal... (22 слов)


# Векторизация и построение FAISS-индекса

In [28]:
EMBED_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
print(f"Загрузка модели эмбеддингов: {EMBED_MODEL_NAME}...")
embed_model = SentenceTransformer(EMBED_MODEL_NAME)
print("Модель загружена.")

chunk_texts = [c["text"] for c in all_chunks]
print("Векторизация чанков")
embeddings = embed_model.encode(chunk_texts, normalize_embeddings=True, show_progress_bar=False)
print(f"Форма эмбеддингов: {embeddings.shape}")

# IndexFlatIP работает с косинусным сходством, если векторы нормализованы
dimension = embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dimension)
faiss_index.add(embeddings.astype(np.float32))
print(f"Индекс FAISS построен. Число векторов: {faiss_index.ntotal}")

Загрузка модели эмбеддингов: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10587.38it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Модель загружена.
Векторизация чанков
Форма эмбеддингов: (10, 384)
Индекс FAISS построен. Число векторов: 10


# Функция retrieval и демонстрация

In [29]:
TOP_K_DEFAULT = 3

def search_faiss(query: str, index: faiss.Index, chunks_list: List[Dict], top_k: int = TOP_K_DEFAULT) -> List[Dict]:
    query_emb = embed_model.encode([query], normalize_embeddings=True).astype(np.float32)
    distances, indices = index.search(query_emb, top_k)
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        if idx != -1:
            results.append({"chunk": chunks_list[idx], "score": float(dist)})
    return results

# Демонстрация поиска на исходной базе
test_queries = ["как сбросить пароль", "правила удаленки", "заказ нового ноутбука"]
print("Тестовый поиск:")
for q in test_queries:
    res = search_faiss(q, faiss_index, all_chunks, TOP_K_DEFAULT)
    print(f"\nЗапрос: '{q}'")
    for i, r in enumerate(res):
        print(f"  {i+1}. [{r['score']:.3f}] {r['chunk']['source_title']}: {r['chunk']['text'][:50]}...")

Тестовый поиск:

Запрос: 'как сбросить пароль'
  1. [0.652] Сброс пароля: Для сброса пароля от корпоративной почты перейдите...
  2. [0.434] Правила информационной безопасности: Запрещено передавать учетные данные третьим лицам....
  3. [0.374] Доступ к базам данных: Доступ к prod-базам строго ограничен. Для dev/stag...

Запрос: 'правила удаленки'
  1. [0.343] Правила информационной безопасности: Запрещено передавать учетные данные третьим лицам....
  2. [0.258] Сброс пароля: Для сброса пароля от корпоративной почты перейдите...
  3. [0.257] Доступ к базам данных: Доступ к prod-базам строго ограничен. Для dev/stag...

Запрос: 'заказ нового ноутбука'
  1. [0.318] Заявка на оборудование: Новое оборудование заказывается через Jira-проект ...
  2. [0.317] Политика удаленной работы: Удаленная работа возможна до 2 дней в неделю по со...
  3. [0.273] Обновление ПО: Обновления ОС и софта пушатся через WSUS и Intune....


# Оценка hit@k и recall@k на контрольных запросах

In [30]:
control_queries = [
    {"query": "как восстановить доступ к почте", "expected": "Сброс пароля"},
    {"query": "нужен ли vpn для работы из дома", "expected": "Настройка VPN"},
    {"query": "сколько дней можно работать удаленно", "expected": "Политика удаленной работы"},
    {"query": "как заказать новый монитор", "expected": "Заявка на оборудование"},
    {"query": "во сколько бэкапы", "expected": "Резервное копирование"},
    {"query": "что делать при потере телефона", "expected": "Правила информационной безопасности"},
    {"query": "номер телефона техподдержки", "expected": "Техподдержка 1-й линии"},
    {"query": "принтер не печатает", "expected": "Настройка принтеров"},
    {"query": "доступ к prod базе", "expected": "Доступ к базам данных"},
    {"query": "обновления windows обязательны", "expected": "Обновление ПО"}
]

def evaluate_retrieval(queries: List[Dict], index: faiss.Index, chunks_list: List[Dict], top_k: int = 3) -> pd.DataFrame:
    records = []
    for q in queries:
        results = search_faiss(q["query"], index, chunks_list, top_k)
        retrieved_titles = [r["chunk"]["source_title"] for r in results]
        hit = 1 if q["expected"] in retrieved_titles else 0
        recall = hit
        records.append({
            "query": q["query"],
            "expected_source": q["expected"],
            "retrieved_sources": ", ".join(retrieved_titles),
            "hit_at_k": hit,
            "recall_at_k": recall
        })
    return pd.DataFrame(records)

# Запуск оценки на исходной базе
eval_df = evaluate_retrieval(control_queries, faiss_index, all_chunks, TOP_K_DEFAULT)
hit_k = eval_df["hit_at_k"].mean()
recall_k = eval_df["recall_at_k"].mean()
print(f"Метрики retrieval (top_k={TOP_K_DEFAULT}):")
print(f"   Hit@{TOP_K_DEFAULT}: {hit_k:.2f}")
print(f"   Recall@{TOP_K_DEFAULT}: {recall_k:.2f}")
print(eval_df.head())

Метрики retrieval (top_k=3):
   Hit@3: 0.80
   Recall@3: 0.80
                                  query            expected_source  \
0       как восстановить доступ к почте               Сброс пароля   
1       нужен ли vpn для работы из дома              Настройка VPN   
2  сколько дней можно работать удаленно  Политика удаленной работы   
3            как заказать новый монитор     Заявка на оборудование   
4                     во сколько бэкапы      Резервное копирование   

                                   retrieved_sources  hit_at_k  recall_at_k  
0  Сброс пароля, Правила информационной безопасно...         1            1  
1  Политика удаленной работы, Техподдержка 1-й ли...         0            0  
2  Политика удаленной работы, Заявка на оборудова...         1            1  
3  Заявка на оборудование, Обновление ПО, Техподд...         1            1  
4  Техподдержка 1-й линии, Заявка на оборудование...         0            0  


# Экспорт оценки в артефакты

In [31]:
eval_df.to_csv(os.path.join(ARTIFACTS_DIR, "retrieval_eval.csv"), index=False)
print("Сохранено: artifacts/retrieval_eval.csv")

Сохранено: artifacts/retrieval_eval.csv


# Сравнение top_k=3 и top_k=5

In [32]:
print("Сравнение параметров retrieval:")
eval_k3 = evaluate_retrieval(control_queries, faiss_index, all_chunks, top_k=3)
eval_k5 = evaluate_retrieval(control_queries, faiss_index, all_chunks, top_k=5)

print(f"top_k=3 -> Hit: {eval_k3['hit_at_k'].mean():.2f}, Recall: {eval_k3['recall_at_k'].mean():.2f}")
print(f"top_k=5 -> Hit: {eval_k5['hit_at_k'].mean():.2f}, Recall: {eval_k5['recall_at_k'].mean():.2f}")
print("Вывод: top_k=5 даёт чуть выше покрытие, но добавляет шум. Для mini-RAG выбираем top_k=3 как оптимальный баланс.")

Сравнение параметров retrieval:


top_k=3 -> Hit: 0.80, Recall: 0.80
top_k=5 -> Hit: 0.90, Recall: 0.90
Вывод: top_k=5 даёт чуть выше покрытие, но добавляет шум. Для mini-RAG выбираем top_k=3 как оптимальный баланс.


# Добавление новых документов и перестройка индекса

In [33]:
new_docs = [
    {"title": "Политика отпусков 2025", "content": "Ежегодный оплачиваемый отпуск составляет 28 календарных дней. Заявление подается через HR-портал минимум за 14 дней. Разделение отпуска на части допускается по согласованию с руководителем."},
    {"title": "Корпоративный мессенджер", "content": "Для внутренней коммуникации используется Mattermost. Каналы создаются по проектам. Личные сообщения шифруются. Запрещено пересылать конфиденциальные данные в публичные каналы."}
]

# Сохраняем результаты ДО обновления
queries_update = ["сколько дней отпуск", "какой мессенджер используем", "как сбросить пароль"]
before_res = [
    ", ".join([r["chunk"]["source_title"] for r in search_faiss(q, faiss_index, all_chunks, 3)])
    for q in queries_update
]

# Обновляем данные и индекс
print("Переиндексация с новыми документами")
updated_docs = raw_documents + new_docs
updated_chunks = []
for doc in updated_docs:
    for chunk in chunk_text(doc["content"], CHUNK_SIZE, CHUNK_OVERLAP):
        updated_chunks.append({"source_title": doc["title"], "text": chunk})

updated_embeddings = embed_model.encode([c["text"] for c in updated_chunks], normalize_embeddings=True).astype(np.float32)
faiss_index_updated = faiss.IndexFlatIP(dimension)
faiss_index_updated.add(updated_embeddings)
print(f"Новый индекс: {faiss_index_updated.ntotal} векторов.")

# Результаты ПОСЛЕ
after_res = []
for q in queries_update:
    q_emb = embed_model.encode([q], normalize_embeddings=True).astype(np.float32)
    _, idxs = faiss_index_updated.search(q_emb, 3)
    sources = [updated_chunks[i]["source_title"] for i in idxs[0] if i != -1]
    after_res.append(", ".join(sources))

Переиндексация с новыми документами
Новый индекс: 12 векторов.


# Экспорт сравнения до/после обновления

In [34]:
update_df = pd.DataFrame({
    "query": queries_update,
    "before_retrieved_sources": before_res,
    "after_retrieved_sources": after_res,
    "changed": [b != a for b, a in zip(before_res, after_res)]
})
update_df.to_csv(os.path.join(ARTIFACTS_DIR, "retrieval_before_after_update.csv"), index=False)
print("Сохранено: artifacts/retrieval_before_after_update.csv")
print(update_df)

Сохранено: artifacts/retrieval_before_after_update.csv
                         query  \
0          сколько дней отпуск   
1  какой мессенджер используем   
2          как сбросить пароль   

                            before_retrieved_sources  \
0  Политика удаленной работы, Обновление ПО, Заяв...   
1  Техподдержка 1-й линии, Резервное копирование,...   
2  Сброс пароля, Правила информационной безопасно...   

                             after_retrieved_sources  changed  
0  Политика отпусков 2025, Политика удаленной раб...     True  
1  Корпоративный мессенджер, Техподдержка 1-й лин...     True  
2  Сброс пароля, Правила информационной безопасно...    False  


# Реализация учебного mini-RAG

In [35]:
def mini_rag(query: str, index: faiss.Index, chunks_list: List[Dict], top_k: int = 3) -> Dict:
    results = search_faiss(query, index, chunks_list, top_k)
    if not results:
        return {"question": query, "answer": "Релевантная информация не найдена.", "retrieved_sources": "None"}
    
    # Убираем дубликаты источников, сохраняя порядок
    sources = list(dict.fromkeys([r["chunk"]["source_title"] for r in results]))
    context = "\n---\n".join([r["chunk"]["text"] for r in results])
    
    best_chunk = results[0]["chunk"]["text"]
    answer = f"Согласно базе знаний: {best_chunk}\n\n(Ответ сформирован на основе найденного фрагмента.)"
    
    return {
        "question": query,
        "answer": answer,
        "retrieved_sources": ", ".join(sources)
    }

print("Тест mini-RAG:")
test_q = "какой мессенджер используется для общения и можно ли слать туда секреты?"
# Передаём обновлённый индекс и обновлённый список чанков
rag_res = mini_rag(test_q, faiss_index_updated, updated_chunks)
print(f"    {rag_res['question']}")
print(f"    {rag_res['answer'][:100]}...")
print(f"    Источники: {rag_res['retrieved_sources']}")

Тест mini-RAG:
    какой мессенджер используется для общения и можно ли слать туда секреты?
    Согласно базе знаний: Для внутренней коммуникации используется Mattermost. Каналы создаются по проек...
    Источники: Корпоративный мессенджер, Правила информационной безопасности, Техподдержка 1-й линии


# Запуск mini-RAG на наборе вопросов и анализ

In [36]:
rag_examples = []
test_questions = [
    "сколько дней отпуск",
    "нужен ли vpn для работы из дома",
    "как восстановить доступ к почте",
    "что делать при потере служебного телефона",
    "какой мессенджер для общения"
]

for q in test_questions:
    rag_examples.append(mini_rag(q, faiss_index_updated, updated_chunks))

rag_df = pd.DataFrame(rag_examples)
print("Примеры работы mini-RAG:")
for _, row in rag_df.iterrows():
    print(f"\n  {row['question']}")
    print(f"    {row['answer'][:120]}...")
    print(f"    {row['retrieved_sources']}")

# Краткий анализ ошибок
print("\nАнализ ограничений:")
print("1. Шаблонный генератор не суммаризирует контекст, а дословно цитирует лучший чанк.")
print("2. При сложных запросах retrieval может выдавать смежные темы (напр., 'потеря телефона' -> 'правила ИБ').")
print("3. Качество ответа линейно зависит от hit@k retrieval: если нужный чанк не попал в top_k, ответ будет неточным.")

Примеры работы mini-RAG:

  сколько дней отпуск
    Согласно базе знаний: Ежегодный оплачиваемый отпуск составляет 28 календарных дней. Заявление подается через HR-портал м...
    Политика отпусков 2025, Политика удаленной работы, Обновление ПО

  нужен ли vpn для работы из дома
    Согласно базе знаний: Удаленная работа возможна до 2 дней в неделю по согласованию с руководителем. Необходимо использов...
    Политика удаленной работы, Техподдержка 1-й линии, Резервное копирование

  как восстановить доступ к почте
    Согласно базе знаний: Для сброса пароля от корпоративной почты перейдите на portal.company.com/reset. Введите логин, пол...
    Сброс пароля, Корпоративный мессенджер, Правила информационной безопасности

  что делать при потере служебного телефона
    Согласно базе знаний: Запрещено передавать учетные данные третьим лицам. Используйте менеджер паролей. При утере устройс...
    Правила информационной безопасности, Техподдержка 1-й линии, Политика удаленной работы

  какой

# Экспорт примеров RAG

In [37]:
rag_df.to_csv(os.path.join(ARTIFACTS_DIR, "rag_examples.csv"), index=False)
print("Сохранено: artifacts/rag_examples.csv")
print(f"Итоговые артефакты: {os.listdir(ARTIFACTS_DIR)}")

Сохранено: artifacts/rag_examples.csv
Итоговые артефакты: ['rag_examples.csv', 'retrieval_before_after_update.csv', 'retrieval_eval.csv']
